### `Task` How dimensionality reduction using Principal Component Analysis (PCA) on the Wine Quality dataset contributes to improving the classification accuracy and efficiency of wine type.

Note : Use KNN for Classification.

Data Link :  [Wine Data](https://docs.google.com/spreadsheets/d/e/2PACX-1vQDVwxneOKOaJL13QMhkAhYrgWlH1tICY7RacUnj_lL8m9uUWaaUf3p7bScNyh_D2Rvt7nc1q11adSy/pub?gid=647503637&single=true&output=csv)

In [26]:
# Data Loading
import pandas as pd
wine_data_path = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQDVwxneOKOaJL13QMhkAhYrgWlH1tICY7RacUnj_lL8m9uUWaaUf3p7bScNyh_D2Rvt7nc1q11adSy/pub?gid=647503637&single=true&output=csv"
df = pd.read_csv(wine_data_path)
df.head(3)

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6


In [27]:
## Percentage of NaN values in each column
nan_percentage = pd.DataFrame({
    'NaN Count': df.isnull().sum(),
    'NaN %': (df.isnull().sum() / len(df)) * 100
})

print(nan_percentage)

                      NaN Count     NaN %
type                          0  0.000000
fixed acidity                10  0.153917
volatile acidity              8  0.123134
citric acid                   3  0.046175
residual sugar                2  0.030783
chlorides                     2  0.030783
free sulfur dioxide           0  0.000000
total sulfur dioxide          0  0.000000
density                       0  0.000000
pH                            9  0.138525
sulphates                     4  0.061567
alcohol                       0  0.000000
quality                       0  0.000000


In [28]:
# Replace NaN values in numerical columns with median

num_cols = df.select_dtypes(include=['number']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

KNN Without PCA

In [29]:
X = df.iloc[:,1:]
y = df.iloc[:,0]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()
knn.fit(X_train,y_train)
y_pred = knn.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)
#Without PCA 93% Accuracy with 784 features + Take more time

0.9361538461538461

KNN WITH PCA

In [16]:
wine_data_path = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQDVwxneOKOaJL13QMhkAhYrgWlH1tICY7RacUnj_lL8m9uUWaaUf3p7bScNyh_D2Rvt7nc1q11adSy/pub?gid=647503637&single=true&output=csv"
df = pd.read_csv(wine_data_path)


In [17]:
# Checking for duplicated data
df.duplicated().sum()
# Dropping Duplicates rows
df.drop_duplicates(inplace=True)
print("Wine Data Shape (After Dropping-) :", df.shape)

Wine Data Shape (After Dropping-) : (5329, 13)


In [18]:
#Step_1-Scaling using Standardisation
'''Why scaling is important before PCA?
PCA is variance-based.
Large-scale features dominate variance.'''
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [19]:
X_train = scaler.fit_transform(X_train) #Fit + Transform testing data
X_test = scaler.transform(X_test) #Only Transfor training data

In [20]:
df.shape

(5329, 13)

In [21]:
#Step_2-PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=5) #Out of 12 features we have taking only 5 features whose eigenvector/eigenvalue is largest means those who explain max variance

In [22]:
X_train_trf = pca.fit_transform(X_train)
X_test_trf = pca.transform(X_test)  #Never fit test data

In [23]:
print(pca.explained_variance_) #Eigenvalues of 61 PC
print(pca.components_) #Eigenvectors=PC directions
print(pca.explained_variance_ratio_) #PC1 explain 25.36% variance of feature space is retained

[3.04490379 2.62390125 1.64258407 1.06743458 0.84228883]
[[-0.25642864 -0.39199169  0.14205699  0.31963015 -0.31578946  0.4239516
   0.47646088 -0.08893553 -0.20457909 -0.30318528 -0.06282153  0.07836034]
 [ 0.26640957  0.10595234  0.14265094  0.34178682  0.26992112  0.10359724
   0.14101162  0.55804951 -0.15495007  0.11897623 -0.49443875 -0.28827876]
 [ 0.46654636 -0.28965121  0.58631822 -0.07843642  0.03060726 -0.09670777
  -0.10233177 -0.04898396 -0.4039049   0.16660934  0.20604991  0.30331588]
 [-0.13558371 -0.06979009  0.04526312  0.14128474  0.14087718  0.29238685
   0.12195539  0.17011983  0.47405418  0.57013582  0.08567626  0.49846094]
 [ 0.15453293  0.16771589 -0.26585348  0.50572821 -0.37443809 -0.25656435
  -0.22635048  0.31024106 -0.06514236 -0.22154355  0.12677239  0.44569311]]
[0.25369316 0.21861636 0.13685567 0.08893577 0.07017723]


In [24]:
knn = KNeighborsClassifier()
knn.fit(X_train_trf,y_train)
y_pred = knn.predict(X_test_trf)
accuracy_score(y_test,y_pred)
#After using PCA 98% Accuracy with only topmost 5 features ,less time

0.9846153846153847